### Script to pull pubmed publications using paperscaper

In [3]:
import numpy as np
import pandas as pd
import polars as pl
import time
import calendar
import calendar
import os

from paperscraper.pubmed import get_pubmed_papers 
from paperscraper.pubmed import get_query_from_keywords_and_date
from datetime import datetime, timedelta
from Bio import Entrez

INFO:paperscraper.load_dumps:Loaded biorxiv dump with 570 entries


In [ ]:
# Keyword list of topics that was pulled from reddit
keywords = ["menstrual cycle","irregular menstruation","menorrhagia", "heavy menstrual bleeding", "abnormal uterine bleeding", "dysmenorrhea", "period cramps",
    "amenorrhea", "breakthrough bleeding","intermenstrual bleeding","combined oral contraceptives","progestin-only pill","IUD","intrauterine device","copper IUD",
    "levonorgestrel IUD","contraceptive side effects","birth control and mood","birth control and libido","tubal ligation","vulvovaginal candidiasis","recurrent yeast infection",
    "bacterial vaginosis","urinary tract infection","UTI","boric acid vaginal suppository","vaginal pH","vaginal microbiome","probiotics vaginal health","pelvic pain",
    "ovarian cyst","ovarian torsion","endometriosis","adenomyosis","pelvic floor dysfunction","pelvic floor physical therapy","vulvodynia","interstitial cystitis",
    "Bartholin cyst","breast lump","fibroadenoma","breast pain","mastalgia", "breast cancer screening","mammography","breast ultrasound","nipple pain","PCOS",
    "polycystic ovary syndrome","hormonal acne","hirsutism","thyroid dysfunction","hypothyroidism","Hashimoto's thyroiditis","perimenopause","menopause hormone therapy",
    "HRT","hot flashes","night sweats","sexually transmitted infection","STI","painful intercourse","dyspareunia","libido","sexual desire","unprotected sex",
    "pregnancy test","medical abortion","misoprostol","mifepristone","abortion access", "hair loss","alopecia","iron deficiency anemia","ferritin","bloating",
    "hemorrhoids","nausea","fatigue","sleep disturbance","heart palpitations","headache","migraine","allergic"]

In [ ]:
Entrez.email = "sebrodnick@willamette.edu"  

output_data = "reddit_keywords_pubmed_1903.csv"

start_date = datetime(1903, 1, 1)
end_date = datetime(2026, 7, 1)

current_date = start_date

while current_date <= end_date:
    last_day = calendar.monthrange(current_date.year, current_date.month)[1]
    start_str = current_date.strftime("%Y/%m/01")
    end_str = current_date.strftime(f"%Y/%m/{last_day}")

    for keyword in keywords:
        month_query = f'("{start_str}"[Date - Create] : "{end_str}"[Date - Create]) AND ({keyword})'
        
        try:
            df = get_pubmed_papers(query = month_query, max_results = 9998)
            
            if df is not None and not df.empty:
                df['keyword'] = keyword

                file_name = os.path.isfile(output_data)
                df.to_csv(output_data, mode='a', header=not file_name, index=False)
                print(f"Saved papers for '{keyword}' for this year {start_str} - {end_str}: {len(df)}")
                del df  # Free up memory after saving to CSV
                
            else:
                print(f"No papers found for '{keyword}' this month {start_str} - {end_str}")
                
        except Exception as e:
            print(f"Error getting papers for '{keyword}': {start_str} - {end_str}: {e}")
            
        time.sleep(1) 
        
    if current_date.month == 12:
        current_date = datetime(current_date.year + 1, 1, 1)
        
    else:
        current_date = datetime(current_date.year, current_date.month + 1, 1)

## Load in both CSVs
### Disclaimer: These two csvs will be deleted, but this is to keep track of the transformation of raw data to final csv

In [4]:
d = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/reddit_keywords_pubmed_1900.csv")
f = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/reddit_keywords_pubmed_1903.csv")

In [31]:
d.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   title     6 non-null      str    
 1   abstract  0 non-null      float64
 2   journal   6 non-null      str    
 3   date      6 non-null      str    
 4   authors   6 non-null      str    
 5   doi       6 non-null      str    
 6   keyword   6 non-null      str    
dtypes: float64(1), str(6)
memory usage: 1.5 KB


In [32]:
f.info()
# 2,814,316

<class 'pandas.DataFrame'>
RangeIndex: 2814316 entries, 0 to 2814315
Data columns (total 7 columns):
 #   Column    Dtype
---  ------    -----
 0   title     str  
 1   abstract  str  
 2   journal   str  
 3   date      str  
 4   authors   str  
 5   doi       str  
 6   keyword   str  
dtypes: str(7)
memory usage: 4.2 GB


In [5]:
pb = pd.concat([d, f], ignore_index=True )

In [73]:
pb.info()
# 2,814,322

<class 'pandas.DataFrame'>
RangeIndex: 2814322 entries, 0 to 2814321
Data columns (total 7 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   title     str   
 1   abstract  object
 2   journal   str   
 3   date      str   
 4   authors   str   
 5   doi       str   
 6   keyword   str   
dtypes: object(1), str(6)
memory usage: 870.7+ MB


## Clean Data

### Duplicates

In [6]:
print(pb[['doi', 'title']].duplicated().sum())
# 579,136

579136


In [7]:
print(pb[['doi', 'title']].value_counts())

doi                                   title                                                                                                                                                                        
10.1016/0010-7824(91)90078-t          An evaluation of the levonorgestrel-releasing IUD: its advantages and disadvantages when compared to the copper-releasing IUDs.                                                  16
10.1186/s12905-026-04368-4            The characterization of patients with premature ovarian failure, endometriosis, and polycystic ovary syndrome in women attending comprehensive health centers in Urmia, Iran.    16
10.3109/07853899309164158             Hormonal intrauterine devices.                                                                                                                                                   15
10.1016/s0031-3955(16)36692-5         Hormonal contraception.                                                                         

In [26]:
pb[pb['doi'] == '10.3109/07853899309164158'].head(5)

,title,abstract,journal,date,authors,doi,keyword
520370,Hormonal intrauterine devices.,Intrauterine administration of progestogens fr...,Annals of medicine,1993-04-01,"['GRybo', 'KAndersson', 'VOdlind']",10.3109/07853899309164158,irregular menstruation
520384,Hormonal intrauterine devices.,Intrauterine administration of progestogens fr...,Annals of medicine,1993-04-01,"['GRybo', 'KAndersson', 'VOdlind']",10.3109/07853899309164158,menorrhagia
520391,Hormonal intrauterine devices.,Intrauterine administration of progestogens fr...,Annals of medicine,1993-04-01,"['GRybo', 'KAndersson', 'VOdlind']",10.3109/07853899309164158,heavy menstrual bleeding
520418,Hormonal intrauterine devices.,Intrauterine administration of progestogens fr...,Annals of medicine,1993-04-01,"['GRybo', 'KAndersson', 'VOdlind']",10.3109/07853899309164158,amenorrhea
520432,Hormonal intrauterine devices.,Intrauterine administration of progestogens fr...,Annals of medicine,1993-04-01,"['GRybo', 'KAndersson', 'VOdlind']",10.3109/07853899309164158,intermenstrual bleeding


Since we are looking at keywords here, I think I am going to keep the studies with multiple keywords. That does mean overall counts for each year will be inflated a little bit. 

I just want to filter for studie sthat have the same doi, title, AND keyword.

In [27]:
print(pb[['doi', 'title', 'keyword']].duplicated().sum())
# 16,372

16372


In [28]:
# Remove duplicates
pb_d = pb.drop_duplicates(subset = ['title', 'doi', 'keyword'], keep='first')

### Fix data issues

In [29]:
print(pb_d['date'].astype(str).str.len().value_counts())

date
10.0    2796054
4.0        1881
31.0          1
61.0          1
Name: count, dtype: int64


In [30]:
pb_d[pb_d['date'].astype(str).str.len().isin([31, 61])]
# These two are messed up
# We need to move everything over to the left from the journal

,title,abstract,journal,date,authors,doi,keyword
2447770,StatPearls,Mammary duct ectasia (MDE) is a non-proliferat...,2026,"['M WalidHamwi', 'RyanWinters']",NaN,nipple pain,NaN
2625294,StatPearls,"Mastalgia, or breast pain, affects up to 70% o...",2026,"['Muhammad T.Tahir', 'Elsa S.Vadakekut', 'Shaf...",NaN,mastalgia,NaN


--> The title is the journal
--> The journal is the date
--> The date is the authors
--> The authors is the doi
--> The doi is the keyword

I have the title for each:
2447770 = Mammary Duct Ectasia
2625294 = Mastalgia

In [31]:
# Change the keyword
pb_d.at[2447770, 'keyword'] = pb_d.at[2447770, 'doi']
pb_d.at[2625294, 'keyword'] = pb_d.at[2625294, 'doi']

In [32]:
pb_d.at[2447770, 'journal'] = pb_d.at[2447770, 'title']
pb_d.at[2625294, 'journal'] = pb_d.at[2625294, 'title']

In [33]:
pb_d.at[2447770, 'authors'] = pb_d.at[2447770, 'date']
pb_d.at[2625294, 'authors'] = pb_d.at[2625294, 'date']

In [34]:
pb_d.at[2447770, 'title'] = 'Mammary Duct Ectasia'
pb_d.at[2625294, 'title'] = 'Mastalgia'

In [35]:
pb_d.at[2447770, 'doi'] = np.nan
pb_d.at[2625294, 'doi'] = np.nan

In [36]:
# Change the date to 2026
pb_d.at[2447770, 'date'] = '2026'
pb_d.at[2625294, 'date'] = '2026'

In [37]:
print(pb_d['date'].astype(str).str.len().value_counts())
# YAY Everything is correct
# Now I need to change the 4 into date time

date
10.0    2796054
4.0        1883
Name: count, dtype: int64


In [38]:
pb_d['date'] = pd.to_datetime(pb_d['date'], format='mixed', errors='coerce')

In [39]:
pb_d.head(2)

,title,abstract,journal,date,authors,doi,keyword
0,IV. Prolapsus of the Rectum in Children.,NaN,Annals of surgery,1900-03-01,['C GCumston'],10.1097/00000658-190001000-00028,hemorrhoids
1,HEMIANOPSIA IN MIGRAINE.,NaN,"Science (New York, N.Y.)",1900-05-04,['J MBaldwin'],10.1126/science.11.279.713-a,migraine


### Create year column

In [40]:
pb_d['year'] = pb_d['date'].dt.year

In [41]:
pb_d.head(2)

,title,abstract,journal,date,authors,doi,keyword,year
0,IV. Prolapsus of the Rectum in Children.,NaN,Annals of surgery,1900-03-01,['C GCumston'],10.1097/00000658-190001000-00028,hemorrhoids,1900.0
1,HEMIANOPSIA IN MIGRAINE.,NaN,"Science (New York, N.Y.)",1900-05-04,['J MBaldwin'],10.1126/science.11.279.713-a,migraine,1900.0


In [43]:
# Change data type from float to integer
pb_d['year'] = pb_d['year'].astype("Int64")

In [44]:
pb_d.info()

<class 'pandas.DataFrame'>
Index: 2797950 entries, 0 to 2814321
Data columns (total 8 columns):
 #   Column    Dtype         
---  ------    -----         
 0   title     str           
 1   abstract  object        
 2   journal   str           
 3   date      datetime64[us]
 4   authors   str           
 5   doi       str           
 6   keyword   str           
 7   year      Int64         
dtypes: Int64(1), datetime64[us](1), object(1), str(5)
memory usage: 885.6+ MB


In [45]:
pb_d.head(2)

,title,abstract,journal,date,authors,doi,keyword,year
0,IV. Prolapsus of the Rectum in Children.,NaN,Annals of surgery,1900-03-01,['C GCumston'],10.1097/00000658-190001000-00028,hemorrhoids,1900
1,HEMIANOPSIA IN MIGRAINE.,NaN,"Science (New York, N.Y.)",1900-05-04,['J MBaldwin'],10.1126/science.11.279.713-a,migraine,1900


In [47]:
pb_d.info()

<class 'pandas.DataFrame'>
Index: 2797950 entries, 0 to 2814321
Data columns (total 8 columns):
 #   Column    Dtype         
---  ------    -----         
 0   title     str           
 1   abstract  object        
 2   journal   str           
 3   date      datetime64[us]
 4   authors   str           
 5   doi       str           
 6   keyword   str           
 7   year      Int64         
dtypes: Int64(1), datetime64[us](1), object(1), str(5)
memory usage: 885.6+ MB


## DataFrame to CSV

In [48]:
pb_d.to_csv("reddit_pubmed_data.csv", index = False)